> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 12 · Notebook 01 — Architecture rules and latency

**Sessions:** S1–S2 (Platform audit against the LLD · Performance engineering) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Extract every import of a module from its syntax tree, including imports hidden in functions.
2. Enforce the layered architecture and a forbidden dependency from the import graph.
3. Read latency percentiles from a Prometheus-style histogram.
4. See what blocking the event loop costs, and keep indicators O(1) per tick.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Let the machine enforce the architecture

The platform's layers, top to bottom: `apps → monitoring → research → strategy → execution → lib → data → domain → core` (`p.LAYERS`). A module may import its own layer or lower ones only, and strategies must never talk to a broker directly. `p.make_toy_package` writes a small `quantforge` package with three planted violations. Step one: every module a file imports, found by walking the whole syntax tree. For `from a.b import c` (absolute, `level == 0`) record both `a.b` and `a.b.c`.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
import ast

def module_imports(path):
    tree = ast.parse(Path(path).read_text())
    out = set()
    for node in ast.walk(tree):                                      # every node, also inside functions
        if isinstance(node, ast.Import):
            out |= {a.name for a in node.names}
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module:
            out.add(node.module)
            out |= {f"{node.module}.{a.name}" for a in node.names}
    return out

import tempfile
pkg = p.make_toy_package(Path(tempfile.mkdtemp()))                   # a small `quantforge` with 3 planted violations
files = sorted(pkg.rglob("*.py"))
mine = {str(f.relative_to(pkg)): p.attempt(lambda f=f: sorted(module_imports(f))) for f in files}
mine = p.check("module_imports", mine, {str(f.relative_to(pkg)): sorted(p.module_imports(f)) for f in files})
{k: v for k, v in mine.items() if v}

In [ ]:
graph = p.import_graph(pkg)
print("layer violations:", p.check_layers(graph, p.LAYERS))
print("strategy → broker:", p.check_forbidden(graph, "quantforge.strategy", ["quantforge.brokers"]))
header_grep = [str(f.relative_to(pkg)) for f in files
               if any(line.startswith(("import quantforge", "from quantforge")) and any(f"quantforge.{x}" in line for x in ("apps", "execution", "brokers"))
                      for line in f.read_text().splitlines())]
print("a grep for top-level import lines finds only:", [f for f in header_grep if f.split("/")[0] in ("lib", "core", "strategy")])

All three violations, including `core/util.py`, whose import of the execution layer hides inside a function where a grep of top-level import lines never looks. In the platform this runs in CI (`import-linter`), so a violation fails the build instead of waiting for a reviewer to notice.

## 2. Measure latency like Prometheus

A histogram keeps **cumulative** counts per bucket (upper bounds 1, 5, 10, 25, 50, 100 ms, +∞). A quantile is read as the upper bound of the first bucket whose cumulative count reaches `q·n`: cheap, mergeable across processes, and conservative. `self.bounds`, `self.counts` (cumulative) and `self.n` are given.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyLatencyHistogram(p.LatencyHistogram):
    def quantile(self, q):
        return next(b for b, c in zip(self.bounds, self.counts) if c >= q * self.n)

latencies = p.latency_samples()                                      # order-ack latencies in seconds, one 400 ms spike
hists = [MyLatencyHistogram(), p.LatencyHistogram()]
for h in hists:
    for x in latencies:
        h.observe(x)
qs = (0.5, 0.9, 0.99, 1.0)
mine = p.check("LatencyHistogram.quantile", [p.attempt(hists[0].quantile, q) for q in qs], [hists[1].quantile(q) for q in qs])
pd.DataFrame({"histogram (bucket upper bound)": mine, "exact percentile": np.quantile(latencies, qs)}, index=[f"p{int(q * 100)}" for q in qs])

The histogram overstates each percentile up to the next bucket bound: choose buckets around your SLO (here 50 ms) so the answer that matters is precise.

## 3. Never block the event loop

The live engine runs on one asyncio loop: market data, orders and heartbeats share it. `p.event_loop_lag` runs a 10 ms ticker while a 200 ms computation runs either **on** the loop or in a thread pool, and reports the worst delay. (Jupyter already runs a loop, so the measurement runs in its own thread.)

In [ ]:
from concurrent.futures import ThreadPoolExecutor

work = lambda: __import__('time').sleep(0.2)                                       # noqa: E731  (stands in for a 200 ms computation)
with ThreadPoolExecutor(1) as pool:
    on_loop = pool.submit(p.event_loop_lag, work, False).result()
    in_executor = pool.submit(p.event_loop_lag, work, True).result()
print(f"worst delay of the 10 ms ticker: {on_loop * 1e3:.0f} ms with the work on the loop, {in_executor * 1e3:.1f} ms with it in an executor")

Everything else waits for the blocking call. In the hot path, also avoid work that grows with history: an EMA can be updated in O(1) per tick, `value += α·(x − value)` with `α = 2/(n + 1)`, starting from the first value. It must equal pandas' `ewm(span=n, adjust=False)` (train/serve parity, as in Part 10).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
class MyStreamingEMA:
    def __init__(self, n):
        self.alpha, self.value = 2 / (n + 1), None

    def update(self, x):
        self.value = x if self.value is None else self.value + self.alpha * (x - self.value)
        return self.value

x = np.cumsum(np.random.default_rng(0).normal(size=500))
live = MyStreamingEMA(20)
mine = p.check("StreamingEMA", p.attempt(lambda: np.array([live.update(v) for v in x])),
               pd.Series(x).ewm(span=20, adjust=False).mean().to_numpy())
print(f"live EMA vs pandas: max difference {np.max(np.abs(mine - pd.Series(x).ewm(span=20, adjust=False).mean().to_numpy())):.1e}")

## Wrap-up

* Architecture rules are checked from the import graph, in CI, over the whole syntax tree.
* Latency is measured as histograms and read as percentiles; buckets sit around the SLO.
* Nothing CPU-heavy on the event loop; O(1) streaming indicators in the hot path.
* Graded version: `labs/part12/week41_integration` (`module_imports`, `import_graph`, `check_layers`, `LatencyHistogram`, `RingBuffer`, `StreamingEMA`, `event_loop_lag`).